# Notebook 2A: Introduction to K-means Clustering with METABRIC

*Authored by Dr. Noelle Anderson in 2026*


## Introduction

In this notebook, you will continue working with the **METABRIC** breast cancer dataset, this time using your scaled data from Notebook 1B to begin **clustering**.

Clustering is a type of **unsupervised learning**, which means there is **no target column** to predict, no "answer" like is a patient sick or healthy to predict. Instead, the goal is to see whether samples naturally group together based on their measured features.

Clustering can be useful for finding patterns that are not obvious at first glance, like whether some patients have similar molecular profiles, whether samples seem to fall into distinct subgroups, or whether there may be hidden structure in a dataset worth studying more closely. In biology and medicine, this can help researchers generate new hypotheses, explore possible disease subtypes, or identify groups of samples that may deserve further investigation.

Today, the focus is on learning the basic mechanics of **K-means clustering** and getting comfortable with how we inspect and visualize clustering results. You will fit K-means models with two different values of **k**, compare the resulting patient groupings, and visualize those group assignments using a **PCA plot**.

This matters because in biology and public health, we are often interested in whether patients, tumors, or samples form meaningful subgroups even before we define an outcome to predict. At the same time, clustering is an exploratory tool, not proof of biological truth, so careful interpretation matters.


By the end of today's notebook, you will be able to:
- load your scaled METABRIC file from Google Drive
- explain what **unsupervised learning** means in this context
- describe at a basic level how **K-means** assigns samples to clusters
- fit K-means models with **`k=2`** and **`k=3`**
- visualize cluster assignments in a PCA plot
- compare two different clustering results and reflect on what clustering can and cannot tell us

*As you work, focus on what the algorithm is doing and how the visualizations help you interpret the result.*


## Unsupervised learning, PCA, and K-means

In **Notebook 1B**, you used **PCA** as an unsupervised method for **summarizing** and **visualizing** variation in a high-dimensional dataset. PCA creates new axes, called principal components, that capture major patterns of variation across the data.

**K-means clustering is different!** It does not create new axes, instead it tries to divide samples into **groups** (**clusters**), based on how similar they are to one another across the original features.

The key differences are:
- **PCA** *helps us summarize complex data by creating new axes that capture the biggest patterns of variation.*
- **K-means** *helps us group samples by assigning each one to a cluster of similar samples.*

In this notebook, we will use PCA only as a visualization tool so we can see the K-means cluster assignments on a 2D plot since when you have >2 features, visualizing data becomes difficult. It's important to understand that the clustering itself is still being done using the full scaled dataset, the data columns themselves, not using only the two PCA axes.


## Notebook 2A roadmap

In this notebook, we will follow this workflow:

1. **Load the scaled METABRIC data from 1B**
2. **Review the dataset structure**
3. **Remind ourselves why scaling matters for K-means**
4. **Learn the basic idea behind K-means clustering**
5. **Fit a K-means model with `k=2`**
6. **Visualize the clusters in PCA space**
7. **Fit a second K-means model with `k=3`**
8. **Compare the two clustering results**
9. **Reflect on what clustering can and cannot tell us**


## Step 0: Import needed packages


In [ ]:
# These are packages you have already seen in earlier notebooks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Our sklearn tools
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans # New today!

You used **PCA** in Notebook 1B, so one part of today's code will look familiar. The new sklearn tool is **`KMeans`**, which is a clustering algorithm. A good pattern to notice in sklearn is that many tools use a similar workflow:
- create the model object
- fit it to the data
- inspect the results

You will see that pattern again today with clustering and with PCA.


## Step 1: Load the scaled METABRIC data from Google Drive

At the end of Notebook 1B, you saved your scaled dataset to your Google Drive folder called `SCIP2026ML`. In this notebook, we will load that file so we can continue from the same starting point.

We will again set `patient_id` as the index so each row stays labeled by patient.


### <font color=0D9488>**Question 1:**</font> Mount your Google Drive and load in your `metabric_1b_scaled.csv` file from the `SCIP2026ML` folder. Make sure `patient_id` is the index and save the data under the variable name `df`.


In [ ]:
# Your solution here!


## Step 2: Review the dataset structure

Before we cluster anything, let's do a quick sanity check. This helps us confirm that the file loaded correctly and that the dataset still looks like the one we expect.


### <font color=0D9488>**Question 2:**</font> Print the shape of the dataframe, the index name, the first 5 rows, and the data types of the first 10 columns so you can confirm the dataset loaded correctly.


In [ ]:
# Your solution here!


A few things should be true here:
- `patient_id` should be the index, not a feature used for clustering
- all remaining columns should be numeric
- the values should already be scaled from Notebook 1B

These details matter because **K-means groups samples based on distance**. Here, distance means a numerical measure of how different two samples are across their feature values. If one column has much larger numbers than the others, differences in that one column can outweigh differences in the rest. That would give that feature too much influence over the clustering.


## Step 3: Why scaling still matters here

In Notebook 1B, scaling mattered because PCA is sensitive to the size of the values in each column, and a similar idea applies for clustering.

K-means decides which samples are similar by comparing how far apart they are across the feature columns. If some features have much larger numeric scales than others, those features can dominate the distance calculations, even when they are not the most biologically important.

Because you already scaled the METABRIC dataset, the features are now on a more comparable scale. This helps K-means treat the columns more evenly and makes the resulting clusters easier to interpret.

## Step 4: What K-means is doing

In this notebook, we are using **K-means clustering**, a type of **unsupervised learning**. There is no target column to predict and no known labels (like "Sick" vs "Healthy"). Instead, the goal is to group samples based on how similar their feature values are.

K-means divides the data into **k clusters**, where **k** is a number that we choose ahead of time. In this notebook, we will try a couple of different values of **k** and see how the grouping changes.

Each cluster is represented by a **centroid**. A centroid is the center of a cluster, based on the average feature values of the samples in that cluster. It is not a real patient, just an average location in feature space representing the center of a cluster.

At a simple level, K-means repeats two steps:

1. **Assign** each sample to the nearest centroid  
2. **Update** each centroid so it becomes the new center of the samples assigned to it

It keeps repeating these steps over many iterations until the cluster assignments stop changing very much.


Visually, if we only had 2 features so we could could plot their real values against each other so every dot is a datapoint, we can think of it like the gif below. Below, each data point could be a patient, and the 5 bigger dots represent centroids at k=5, at first randomly chosen, then closest datapoints assigned, then updated over each iteration.

We won't get deeper into the algorithmic details, but I do recommend watching this [8 minute video](https://www.youtube.com/watch?v=4b5d3muPQmA) to understand K-means better.



In [1]:
from IPython.display import Image
Image(url='https://anhui-gui.com/posts/machine-learning/feature.gif')

---

A few important ideas about K-means clustering to keep in mind:
- the cluster labels, like `0`, `1`, and `2`, are just names
- cluster `0` is not better or worse than cluster `1`
- centroid points are not real samples, they are average positions
- once we choose a value for **k**, K-means will always return that many clusters, even if the real data do not separate into perfectly clear groups or if the k number of groups is biologically meaningless.

## Step 5: Fit a K-means model with `k=2`

Now we will run our first K-means model.

A couple new syntax notes:
- `n_clusters=2` tells sklearn to build **2 clusters**
- `random_state=42` helps make the result reproducible. The number itself is meaningless, just a starting point for a random number generator, but if we set the same number we should get the same results.

After fitting our clustering model, we can inspect the cluster assignments using `model.labels_`:


In [ ]:
# Create the K-means model for 2 clusters
kmeans_2 = KMeans(n_clusters=2, random_state=42)

# Fit the model to the scaled dataframe— this is where it learns from our data!
kmeans_2.fit(df)

# Look at the first 15 cluster assignments
print('First 15 cluster labels— which clusters did it put each patients in?:')
print(kmeans_2.labels_[:15]) # NOTICE: we use .labels_ to extract the cluster labels per datapoint/patient

# Count how many patients were assigned to each cluster with value_counts()
cluster_counts_2 = pd.Series(kmeans_2.labels_).value_counts()
print('\nCluster counts for k=2— How many patients per cluster?:')
print(cluster_counts_2)


K-means stores the labels in the same row order as the dataframe you fit. That means the first label matches the first patient row, the second label matches the second patient row, and so on.

Notice again that the labels are just cluster names, 0 and 1 don't inherently mean anything. And just because we told it to make 2 clusters doesn't mean these 2 groups are biologically meaningful, they are just the algorithm's best attempt to separate the data into 2 groups. We will see in the next notebook how to choose the value of k.

## Step 6: Visualize the `k=2` clusters in PCA space

The METABRIC dataset has many columns, so each sample is described using many measured features. That means the data exist in more than 2 or 3 dimensions, which we cannot visualize directly in a simple plot. A regular scatterplot can only display 2 axes at once, and 3D and beyond plots are difficult to interpret visually, so we cannot show the full feature space that K-means is using.

To make the clustering easier to visualize, we can use **PCA** to create a 2D summary of the data colored by the K-means assigned clusters. This does not show the full dataset exactly, but it gives us a useful simplified view.

This is an important distinction:
- **K-means** is still clustering using the full scaled dataset, across all feature columns
- **PCA** is only helping us visualize those assignments on a 2D plot by summarizing the data into two principal components

If clusters overlap somewhat in the PCA plot, that does not automatically mean the clustering is wrong. It may just mean that two PCA axes cannot show every detail from the full high-dimensional space.


In [ ]:
# Run PCA with 2 components so we can make a 2D plot
pca = PCA(n_components=2)
X_pca = pca.fit_transform(df)

# Put the PCA coordinates into a dataframe
pca_df = pd.DataFrame(X_pca, index=df.index, columns=['PC1', 'PC2'])

Now let's use that pca_df to make a scatter plot of `PC1` vs `PC2` and color the points by the `k=2` cluster labels. Notice how we color the points with `hue` and the cluster labels.


In [ ]:
# Add the cluster labels from k=2 so we can plot them
pca_df['cluster_k2'] = kmeans_2.labels_

# Now plot!
plt.figure(figsize=(7, 5))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='cluster_k2', alpha=0.8)
# alpha makes points transparent so they're easier to view

plt.title('K-means Clusters on METABRIC Data (k=2)')
plt.legend(title='Cluster')
plt.show()


When you look at this plot, ask yourself:
- Do the two clusters look mostly separated, or is there noticeable overlap?
- Does one side of the plot seem dominated by one cluster more than the other?
- Do the clusters look very compact, or somewhat spread out?

Remember that PCA is only showing a simplified 2D view of the full feature space.



### <font color=0D9488>**Question 3:**</font> Look at your `k=2` PCA plot. Describe what you notice about the separation and overlap between the two clusters.


**Your solution here!**


## Step 7: Fit a second K-means model with `k=3`

Now we will repeat the same clustering process, but this time we will ask K-means for **3 clusters** instead of 2.

This is a useful comparison because changing **k** changes the grouping the algorithm is forced to create. In other words, the model is now being asked to divide the same dataset into three groups instead of two.


### <font color=0D9488>**Question 4:**</font> Fit a K-means model now with 3 clusters called `kmeans_3`; and use `random_state=42`. Then print the first 15 patient labels and the number of patients in each cluster.


In [ ]:
# Your solution here!


### <font color=0D9488>**Question 5:**</font> Make a second PCA scatter plot of `PC1` vs `PC2`, this time coloring the points by the `k=3` cluster labels.

*Hint 1: no need to re-run the PCA, it doesn't change based on the clustering, here we are just changing the coloring and labels. You will need to add a column to the original `pca_df` representing the k=3 cluster labels*

*Hint 2: If your dots are hard to see with the default color palette, try adding `palette="deep"` to the scatterplot command.*


In [ ]:
# Your solution here!

A useful question here is whether the `k=3` result looks like one of the earlier `k=2` groups has now been split into smaller pieces, or whether the new pattern looks different in another way.

Also remember that cluster label numbers do **not** carry over across different models. Cluster `0` from the `k=2` model is not automatically the same thing as cluster `0` from the `k=3` model, those are just arbitrary numbered labels.


### <font color=0D9488>**Question 6:**</font> Compare the `k=2` and `k=3` PCA plots. Describe at least two differences you notice.


**Your solution here!**


## Step 8: What clustering can and cannot tell us

Clustering can help us explore complex biomedical data by looking for groups of patients or samples that seem similar to each other.

At the same time, clustering does not prove that these groups are real biological categories. It only shows patterns based on the data and the method we used.

Here are a few important limits to remember:

- **K-means always makes clusters.** If we ask for 2 or 3 clusters, it will return them even if the groups are not clearly separated.
- **The results can change.** Different variables, scaling choices, or preprocessing steps can lead to different clusters.
- **Clusters are not automatically meaningful.** A mathematical grouping does not always match a real biological subtype.
- **Real patient patterns may overlap.** In biomedical data, people do not always fit into neat, separate groups.

In cancer datasets like our METABRIC data, clustering can sometimes help researchers look for possible tumor subtypes or patient subgroups. That can be useful for generating hypotheses, but it can also be misleading if the data are incomplete, noisy, or not representative of all patient populations.

That means clustering should be treated as a starting point for further analysis, not as final truth.

### <font color=0D9488>**Question 7:**</font> Briefly explain one similarity and one difference between PCA and K-means. Then describe one reason we should be cautious when interpreting clusters in real patient data.


**Your solution here!**


# Congratulations, you have completed today's notebook!


## Key Takeaways

In this notebook, you:
- loaded the scaled METABRIC dataset from Notebook 1B
- reviewed what **unsupervised learning** means
- distinguished **PCA** from **K-means clustering**
- learned that K-means groups samples by similarity using **centroids**
- fit K-means models with **`k=2`** and **`k=3`**
- used **PCA** to visualize the cluster assignments in 2 dimensions
- compared how the clustering changed when you changed the value of **k**
- reflected on why clustering is useful, but also why it requires caution


In this notebook, you took your first step into unsupervised learning by using K-means clustering to group similar patient samples without using a known outcome variable. You also saw that PCA and K-means serve different purposes: PCA helps us visualize high-dimensional data, while K-means tries to divide the data into groups based on similarity. Most importantly, you practiced treating clustering as a tool for exploring patterns, not as automatic proof that clear biological subtypes exist.

## Where This Fits in the ML Workflow

This notebook continued the unsupervised machine learning workflow:

1. In **Notebook 1A**, you cleaned and prepared the METABRIC dataset
2. In **Notebook 1B**, you scaled the data and used **PCA** to summarize and visualize structure
3. Here in **Notebook 2A**, you used the scaled data to begin **clustering**, which asks whether similar patients can be grouped together

So far, the workflow looks like this:
- **load data**
- **explore data**
- **preprocess data**
- **summarize structure with PCA**
- **group samples with clustering**

In the next notebook, you will go further by thinking more carefully about how to choose `k` and evaluate and compare clustering results.
